# Final Project — Databricks Fundamentals

> **Duration:** ~45 min | **Level:** Beginner
> **Goal:** Apply everything from the training in one end-to-end pipeline

## What you will build

```
CSV / JSON files
   → [1] Load  → [2] Explore  → [3] Transform  → [4] Delta Lake  → [5] Workflow
```

| # | Topic | Module |
|---|---|---|
| 1 | Reading CSV and JSON files | M02 |
| 2 | Exploration: `describe`, `summary` | M02 |
| 3 | `withColumn`, `cast`, `when`/`otherwise` | M02 |
| 4 | Writing to Delta Lake (managed tables) | M03 |
| 5 | Scheduling with Databricks Workflows | M10 |

---

## Step 0 — Setup

Run the shared setup notebook to configure catalog/schema names and dataset paths.

In [ ]:
%run ../../setup/00_setup

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType
)
from pyspark.sql.functions import col, when, to_date, upper, concat, lit, count

CUSTOMERS_CSV = f"{DATASET_PATH}/customers/customers.csv"
ORDERS_JSON   = f"{DATASET_PATH}/orders/orders_batch.json"
print(f'customers: {CUSTOMERS_CSV}')
print(f'orders:    {ORDERS_JSON}')

---

## Step 1 — Load Raw Data

Load the files using **manual schema** — always preferred in production to avoid full-file scans.

In [ ]:
customers_schema = StructType([
    StructField("customer_id",       StringType(),  False),
    StructField("first_name",        StringType(),  True),
    StructField("last_name",         StringType(),  True),
    StructField("email",             StringType(),  True),
    StructField("country",           StringType(),  True),
    StructField("city",              StringType(),  True),
    StructField("age",               IntegerType(), True),
    StructField("credit_score",      IntegerType(), True),
    StructField("registration_date", StringType(),  True),
])

customers_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .schema(customers_schema)
    .csv(CUSTOMERS_CSV)
)
print(f'Customers loaded: {customers_raw.count()} rows')
customers_raw.printSchema()

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  False),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("quantity",       IntegerType(), True),
    StructField("total_amount",   FloatType(),   True),
    StructField("order_date",     StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("status",         StringType(),  True),
])

orders_raw = (
    spark.read
    .schema(orders_schema)
    .json(ORDERS_JSON)
)
print(f'Orders loaded: {orders_raw.count()} rows')
orders_raw.printSchema()

---

## Step 2 — Explore the Data

Before transforming, always **understand your data**.

| Method | What it shows |
|---|---|
| `display(df)` | Interactive Databricks table viewer |
| `df.describe()` | count, mean, std, min, max for numeric columns |
| `df.summary()` | Same + 25th / 50th / 75th percentiles |

In [ ]:
display(customers_raw.limit(5))

In [ ]:
print("=== customers: describe ===")
customers_raw.describe("age", "credit_score").show()

print("\n=== orders: summary ===")
orders_raw.select("total_amount", "quantity").summary().show()

In [ ]:
# Check for nulls
null_counts = customers_raw.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customers_raw.columns
])
print("Null values per column:")
null_counts.show()

---

## Step 3 — Transform the Data

| Transformation | Purpose |
|---|---|
| `cast()` | Convert date string → DateType |
| `withColumn` | Add `full_name` column |
| `when`/`otherwise` | Classify credit segment and order tier |
| `filter` | Drop rows with missing critical fields |

In [ ]:
customers_clean = (
    customers_raw
    .withColumn("registration_date", to_date(col("registration_date"), "yyyy-MM-dd"))
    .withColumn("full_name", concat(col("first_name"), lit(" "), col("last_name")))
    .withColumn(
        "credit_segment",
        when(col("credit_score") >= 750, "Excellent")
        .when(col("credit_score") >= 700, "Good")
        .when(col("credit_score") >= 650, "Fair")
        .otherwise("Poor")
    )
    .filter(col("customer_id").isNotNull())
)
print(f'Customers after transform: {customers_clean.count()} rows')
display(customers_clean.select(
    "customer_id", "full_name", "country", "credit_score", "credit_segment", "registration_date"
).limit(10))

In [ ]:
orders_clean = (
    orders_raw
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
    .withColumn(
        "order_tier",
        when(col("total_amount") >= 500, "Premium")
        .when(col("total_amount") >= 200, "Standard")
        .otherwise("Small")
    )
    .withColumn("payment_method", upper(col("payment_method")))
    .filter(col("order_id").isNotNull())
)
print(f'Orders after transform: {orders_clean.count()} rows')
display(orders_clean.select(
    "order_id", "customer_id", "total_amount", "order_tier", "payment_method", "order_date"
).limit(10))

---

## Step 4 — Write to Delta Lake (Bronze Layer)

```
catalog
└── bronze
    ├── lab_customers   ← customers_clean
    └── lab_orders      ← orders_clean
```

Use `mode("overwrite")` — safe for an initial load or full refresh.

In [ ]:
(
    customers_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.lab_customers")
)
count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.lab_customers").count()
print(f'lab_customers written: {count} rows')

In [ ]:
(
    orders_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.lab_orders")
)
count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.lab_orders").count()
print(f'lab_orders written: {count} rows')

In [ ]:
print("Delta history — lab_customers:")
spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.lab_customers").select(
    "version", "timestamp", "operation"
).show(5, truncate=False)

---

## Step 5 — Schedule as a Databricks Workflow

### Instructions (UI)

1. Open **Workflows** tab in the left sidebar
2. Click **Create Job**
3. Set:
   - **Job name:** `lab-final-pipeline`
   - **Task name:** `run_lab_final`
   - **Type:** `Notebook`
   - **Source:** this notebook (`LAB_FINAL`)
   - **Cluster:** Job cluster (cheapest) or existing cluster
4. Set **Schedule:** daily at 06:00 AM UTC
5. Click **Create** → **Run now** to test

| Concept | What it means |
|---|---|
| **Job cluster** | Starts fresh per run, auto-terminates → cheaper |
| **CRON expression** | `0 6 * * *` = every day at 06:00 |
| **Email alerts** | Set in **Notifications** tab |

---

## Optional — Mini Analytics

Explore the data you just loaded with SQL queries.

In [ ]:
spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.lab_customers").createOrReplaceTempView("customers")
spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.lab_orders").createOrReplaceTempView("orders")

In [ ]:
display(spark.sql("""
    SELECT country, COUNT(*) AS customer_count
    FROM customers
    GROUP BY country ORDER BY customer_count DESC LIMIT 5
"""))

In [ ]:
display(spark.sql("""
    SELECT order_tier,
           COUNT(*) AS order_count,
           ROUND(SUM(total_amount), 2) AS total_revenue
    FROM orders
    GROUP BY order_tier ORDER BY total_revenue DESC
"""))

---

## Summary & Checklist

- [ ] Loaded `customers.csv` and `orders_batch.json` with manual schema
- [ ] Explored data with `describe()` / `summary()` / `display()`
- [ ] Applied `cast()`, `withColumn()`, `when`/`otherwise` transformations
- [ ] Wrote DataFrames to Delta Lake as managed tables
- [ ] Verified the Delta tables with `DESCRIBE HISTORY`
- [ ] Created a Databricks Workflow to schedule this pipeline

---

← [Workflows & Automation](../modules/M10_orchestration_jobs.ipynb) | [Intro](../modules/M00_intro.ipynb) →